<a href="https://colab.research.google.com/github/cristiangromero/LGRT-PIA/blob/main/PIA_Activiad_N6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
class RedBayesianaTelefono:
    """Implementación de red bayesiana para el problema del teléfono"""

    def __init__(self):
        # Probabilidades a priori
        self.p_bateria_baja = 0.25
        self.p_sin_senal = 0.15

        # P(NS=Sí | Batería Baja, Sin Señal)
        self.cpt_no_suena = {
            (True, True): 0.95,
            (True, False): 0.80,
            (False, True): 0.85,
            (False, False): 0.05
        }

    def probabilidad_conjunta(
        self, bateria: bool, senal: bool, no_suena: bool
    ) -> float:

        p_b = self.p_bateria_baja if bateria else (1 - self.p_bateria_baja)
        p_s = self.p_sin_senal if senal else (1 - self.p_sin_senal)

        p_ns = self.cpt_no_suena[(bateria, senal)]

        if not no_suena:
            p_ns = 1 - p_ns

        return p_b * p_s * p_ns

    def inferencia_exacta(self, evidencia: dict) -> dict:

        estados = [
            (b, s, ns)
            for b in [True, False]
            for s in [True, False]
            for ns in [True, False]
        ]

        estados_consistentes = []

        for b, s, ns in estados:

            estado = {
                "bateria": b,
                "senal": s,
                "no_suena": ns
            }

            if all(estado[k] == v for k, v in evidencia.items()):
                prob = self.probabilidad_conjunta(b, s, ns)
                estados_consistentes.append((estado, prob))

        total_evidencia = sum(
            prob for _, prob in estados_consistentes
        )

        p_bateria = sum(
            prob
            for estado, prob in estados_consistentes
            if estado["bateria"]
        ) / total_evidencia

        p_senal = sum(
            prob
            for estado, prob in estados_consistentes
            if estado["senal"]
        ) / total_evidencia

        return {
            "P(Batería Baja = Sí | evidencia)": p_bateria,
            "P(Sin Señal = Sí | evidencia)": p_senal,
            "P(Evidencia Total)": total_evidencia
        }


# Crear la red
red = RedBayesianaTelefono()

# Ejercicio 1: el teléfono NO suena
res1 = red.inferencia_exacta({"no_suena": True})

print(
    f"Ejercicio 1 - P(Batería Baja | No Suena) = "
    f"{res1['P(Batería Baja = Sí | evidencia)']:.4f} "
    f"({res1['P(Batería Baja = Sí | evidencia)'] * 100:.2f}%)"
)

# Ejercicio 2: el teléfono SÍ suena
res2 = red.inferencia_exacta({"no_suena": False})

print(
    f"Ejercicio 2 - P(Sin Señal | Sí Suena) = "
    f"{res2['P(Sin Señal = Sí | evidencia)']:.4f} "
    f"({res2['P(Sin Señal = Sí | evidencia)'] * 100:.2f}%)"
)

Ejercicio 1 - P(Batería Baja | No Suena) = 0.6173 (61.73%)
Ejercicio 2 - P(Sin Señal | Sí Suena) = 0.0281 (2.81%)
